
# Fusion des données RPPS et agrégation de l'offre de soins par commune

Ce notebook fusionne la table personne-activité avec la table savoir-faire, puis agrège l'offre de soins au niveau communal.

**Objectif :**
- charger les tables RPPS préparées ;
- fusionner les informations sur les professionnels et leurs savoir-faire ;
- compter le nombre de professionnels distincts par commune ;
- produire une table d'offre de soins communale.


In [ ]:
import pandas as pd

In [ ]:

personne = pd.read_csv(
    "s3://mangane/Projet_PDS_2026/processed/personne_activite_clean.csv",
    dtype={"Identifiant PP": str}
)

savoirfaire = pd.read_csv(
    "s3://mangane/Projet_PDS_2026/processed/savoirfaire_clean.csv",
    dtype={"Identifiant PP": str}
)


In [ ]:

table_finale = pd.merge(
    personne,
    savoirfaire,
    on="Identifiant PP",
    how="left"
)

print("Dimensions de la table finale :")
print(table_finale.shape)

print("\n10 premières lignes :")
print(table_finale.head(10).to_string())


In [ ]:

offre_commune = (
    table_finale
    .groupby(
        [
            "Code commune (coord. structure)",
            "Libellé commune (coord. structure)"
        ]
    )
    .agg(
        nb_professionnels=("Identifiant PP", "nunique")
    )
    .reset_index()
)

print("\nDimensions de la table par commune :")
print(offre_commune.shape)

print("\n10 premières communes :")
print(offre_commune.head(10).to_string())


In [ ]:

offre_commune.to_csv(
    "s3://mangane/Projet_PDS_2026/processed/offre_soins_commune.csv",
    index=False
)

print("\nFichier enregistré : offre_soins_commune.csv")
